# Pydantic AI + Neo4j GraphRAG

The [main notebook](./pydantic_ai.ipynb) gave an agent Cypher. That answers questions about things
the graph stores as structure - who invested in whom, who competes with whom. It cannot answer
"what are people saying about data centre demand", because that lives in text.

This notebook adds retrieval. [`neo4j-graphrag`](https://neo4j.com/docs/neo4j-graphrag-python/)
searches vector and full-text indexes, and - this is the part that makes it graph RAG rather than
vector search - lets a Cypher query run on top of every hit, so each retrieved chunk arrives with
the article it came from and the companies that article mentions.

| Section | What it shows |
| --- | --- |
| 2 | The text side of the graph, and matching an embedder to an index |
| 3 | `VectorRetriever` - plain semantic search |
| 4 | `VectorCypherRetriever` - every hit expanded through the graph |
| 5 | `HybridCypherRetriever` - vector plus full-text, for names and numbers |
| 6 | The three compared on one question |
| 7 | Retrievers as typed agent tools, with cited structured output |
| 8 | Letting the agent pick a retriever, and seeing which it picked |
| 9 | The `GraphRAG` pipeline, for when you do not need an agent at all |

Runs against `companies`, the public Neo4j demo database, which carries ~65k news articles already
chunked and embedded.

## 1. Setup

`neo4j-graphrag` 1.21 or newer matters here: the retrievers emit Neo4j's newer `SEARCH` clause when
the server supports it and fall back to the vector procedure when it does not. On older versions
that fallback is missing and queries against a Cypher 5 server fail outright.

In [1]:
%pip install -q --upgrade "pydantic-ai>=2.0" "neo4j-graphrag>=1.21" neo4j openai


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import os
from dotenv import load_dotenv
load_dotenv()

NEO4J_URI = "neo4j+s://demo.neo4jlabs.com"
NEO4J_USERNAME = "companies"
NEO4J_PASSWORD = "companies"
NEO4J_DATABASE = "companies"

MODEL = "openai:gpt-5.4-mini"

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

os.environ["PYDANTIC_AI_NO_BANNER"] = "1"

print("Configured.")

Configured.


## 2. The text side of the graph

`neo4j-graphrag` takes a **synchronous** driver - the retrievers are sync, unlike the async driver
the main notebook used. Section 7 bridges that gap in one line when the retrievers become agent
tools.

In [3]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

shape, _, _ = driver.execute_query(
    """
    MATCH (a:Article)-[:HAS_CHUNK]->(c:Chunk)
    WITH a, count(c) AS chunks
    RETURN count(a) AS articles, sum(chunks) AS chunks, avg(chunks) AS chunks_per_article
    """,
    database_=NEO4J_DATABASE,
)
row = shape[0]
print(f"Articles: {row['articles']:,}   Chunks: {row['chunks']:,}   "
      f"Chunks per article: {row['chunks_per_article']:.1f}")

sample, _, _ = driver.execute_query(
    """
    MATCH (a:Article)-[:HAS_CHUNK]->(c:Chunk)
    WHERE a.title IS NOT NULL
    OPTIONAL MATCH (a)-[:MENTIONS]->(o:Organization)
    RETURN a.title AS title, toString(a.date) AS date, a.sentiment AS sentiment,
           collect(DISTINCT o.name)[..4] AS mentions, left(c.text, 160) AS excerpt
    LIMIT 2
    """,
    database_=NEO4J_DATABASE,
)
for record in sample:
    print(f"\n{record['title']}  ({record['date']}, sentiment {record['sentiment']})")
    print(f"  mentions: {', '.join(record['mentions'])}")
    print(f"  {record['excerpt']}...")

Articles: 65,135   Chunks: 108,112   Chunks per article: 1.7

Funds for Consumer Behavior Specialist Infinite Analytics  (2022-03-10T00:00:00Z, sentiment 0.856)
  mentions: Infinite Analytics
  Boston and Mumbai-based consumer behavior analyses platform Infinite Analytics has raised an undisclosed sum of investment from a syndicate of investors, which ...

AI in Retail Market is Expected to Witness a Sustainable Growth Over 2021-2030  (2021-12-03T13:14:00Z, sentiment 0.908)
  mentions: Infinite Analytics
  Market is driven by factors such as perpetually growing internet users and smart devices, increasing awareness about AI and big data & analytics.
PORTLAND, PORT...


### Which indexes exist

A chunk can be embedded several times over, by different models. Each one gets its own index, and
they are not interchangeable.

In [4]:
indexes, _, _ = driver.execute_query(
    """
    SHOW VECTOR INDEXES
    YIELD name, labelsOrTypes, properties, options
    RETURN name, labelsOrTypes[0] AS label, properties[0] AS property,
           options.indexConfig['vector.dimensions'] AS dimensions
    ORDER BY name
    """,
    database_=NEO4J_DATABASE,
)
print(f"{'Index':<20}{'Label':<12}{'Property':<22}{'Dimensions':>11}")
for record in indexes:
    print(f"{record['name']:<20}{record['label']:<12}{record['property']:<22}{record['dimensions']:>11}")

fulltext, _, _ = driver.execute_query(
    "SHOW FULLTEXT INDEXES YIELD name, labelsOrTypes, properties RETURN name, labelsOrTypes, properties",
    database_=NEO4J_DATABASE,
)
print("\nFull-text indexes:")
for record in fulltext:
    print(f"  {record['name']}  on {record['labelsOrTypes']}{record['properties']}")

Index               Label       Property               Dimensions
fewshot             Fewshot     embedding                    1536
news                Chunk       embedding                    1536
news_google         Chunk       embedding_google              768
news_google_004     Chunk       embedding_google_004          768
news_sbert          Chunk       embedding_sbert               384

Full-text indexes:
  entity  on ['Person', 'Organization']['name']
  news_fulltext  on ['Chunk']['text']


### Match the embedder to the index

In [ ]:
INDEX_NAME = "news"
FULLTEXT_INDEX_NAME = "news_fulltext"

from neo4j_graphrag.embeddings import OpenAIEmbeddings

embedder = OpenAIEmbeddings()

meta, _, _ = driver.execute_query(
    """
    SHOW VECTOR INDEXES YIELD name, labelsOrTypes, properties
    WHERE name = $name
    RETURN labelsOrTypes[0] AS label, properties[0] AS property
    """,
    name=INDEX_NAME, database_=NEO4J_DATABASE,
)
label, embedding_property = meta[0]["label"], meta[0]["property"]

probe, _, _ = driver.execute_query(
    f"""
    MATCH (n:{label})
    WHERE n.`{embedding_property}` IS NOT NULL AND n.text IS NOT NULL
    RETURN n.text AS text, n.`{embedding_property}` AS stored
    LIMIT 1
    """,
    database_=NEO4J_DATABASE,
)

stored = probe[0]["stored"]
fresh = embedder.embed_query(probe[0]["text"])

dot = sum(a * b for a, b in zip(fresh, stored))
norm = (sum(a * a for a in fresh) ** 0.5) * (sum(b * b for b in stored) ** 0.5)
similarity = dot / norm

print(f"Index '{INDEX_NAME}' on ({label}.{embedding_property}), {len(stored)} dimensions")
print(f"Self-similarity: {similarity:.3f}")
assert similarity > 0.95, (
    "Embedder does not match this index. Every search below would return quietly wrong "
    "results. Pick the index built by this model, or the model that built this index."
)
print("Embedder matches the index ✓")

/usr/local/python/3.12.1/lib/python3.12/site-packages/google/auth/transport/grpc.py:44: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In October 2026, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


Index 'news' on (Chunk.embedding), 1536 dimensions
Self-similarity: 1.000
Embedder matches the index ✓


## 3. `VectorRetriever` - plain semantic search

The baseline: embed the question, find the nearest chunks, return their text. No graph involved.

In [6]:
from neo4j_graphrag.retrievers import VectorRetriever

vector_retriever = VectorRetriever(
    driver=driver,
    index_name=INDEX_NAME,
    embedder=embedder,
    return_properties=["text"],
    neo4j_database=NEO4J_DATABASE,
)

results = vector_retriever.search(query_text="growth in cloud infrastructure spending", top_k=3)

for item in results.items:
    print(f"score {item.metadata['score']:.3f}  {str(item.content)[:150]}...")
    print()

score 0.929  {'text': ' deals dating back to 2020. In April, IBM bought Taos, a provider of managed and professional IT services with a focus on public cloud compu...

score 0.929  {'text': 'New Zealand\'s infrastructure-as-a-service (IaaS) market grew by 27.7 per cent in 2021 to a total of NZ$576 million, according to analyst fi...

score 0.926  {'text': 'China has borne less of the brunt of economic recession than we have felt here in the States. In a position to invest capital into the futur...



## 4. `VectorCypherRetriever` - expand every hit through the graph

Same vector search, but a Cypher query runs afterwards with each matched node bound to `node` and
its similarity to `score`. Whatever that query returns is the context the model sees.

This is where a graph earns its place in RAG. A chunk on its own is a paragraph with no provenance.
One hop out and it arrives with its article, the publication date, the sentiment, the companies the
article mentions and - one more hop - the competitors of those companies. The model can then answer
questions the chunk alone does not contain.

`result_formatter` turns each returned record into a `RetrieverResultItem`. Skip it and you get the
driver's raw record repr, which is awkward to parse and worse to put in a prompt.

In [7]:
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem

RETRIEVAL_QUERY = """
WITH node AS chunk, score
MATCH (article:Article)-[:HAS_CHUNK]->(chunk)
OPTIONAL MATCH (article)-[:MENTIONS]->(org:Organization)
OPTIONAL MATCH (org)-[:HAS_COMPETITOR]->(rival:Organization)
RETURN chunk.text AS text,
       article.id AS article_id,
       article.title AS title,
       toString(article.date) AS date,
       article.sentiment AS sentiment,
       collect(DISTINCT org.name)[..5] AS companies,
       collect(DISTINCT rival.name)[..5] AS competitors,
       score
ORDER BY score DESC
"""


def format_with_context(record) -> RetrieverResultItem:
    """One retrieved chunk, rendered as something worth putting in a prompt."""
    return RetrieverResultItem(
        content=(
            f"{record['title']} ({record['date']})\n"
            f"Companies: {', '.join(record['companies']) or 'none recorded'}\n"
            f"Competitors of those: {', '.join(record['competitors']) or 'none recorded'}\n"
            f"{record['text']}"
        ),
        metadata={
            "article_id": record["article_id"],
            "title": record["title"],
            "date": record["date"],
            "sentiment": record["sentiment"],
            "companies": record["companies"],
            "score": record["score"],
        },
    )


graph_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=INDEX_NAME,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_with_context,
    neo4j_database=NEO4J_DATABASE,
)

results = graph_retriever.search(query_text="growth in cloud infrastructure spending", top_k=3)

for item in results.items:
    print(item.content[:320])
    print("-" * 70)

IBM acquires container services provider BoxBoat (2021-07-08T07:00:00Z)
Companies: Red Hat, Boxboat Technologies, Kubernetes, IBM
Competitors of those: Dell, Swainson, Super Micro Computer, Inc., NCR, Vzaar
 deals dating back to 2020. In April, IBM bought Taos, a provider of managed and professional IT services with a 
----------------------------------------------------------------------
Spark leads $576M NZ IaaS market (2022-06-03T09:05:00Z)
Companies: Huawei, Spark New Zealand, Gartner, Microsoft, Google
Competitors of those: Forrester Research, SolutionMap, IDC, Dell, Wipro
New Zealand's infrastructure-as-a-service (IaaS) market grew by 27.7 per cent in 2021 to a total of NZ$576 million, according t
----------------------------------------------------------------------
China’s Cloud Positioning and the Prospect of Foreign Outsourcing (2013-01-18T00:00:00Z)
Companies: VanceInfo Technologies, Hewlett Packard Enterprise, Forbes
Competitors of those: Wipro Ltd, DCM, Infosys, Chinasoft 

## 5. `HybridCypherRetriever` - vector plus full-text

Vector search is good at meaning and bad at strings. Ask it about a product code, a ticker or an
unusual company name and the nearest neighbours are about the general topic rather than that exact
term.

Hybrid retrieval runs both indexes and merges the rankings, then applies the same Cypher expansion.
Names and numbers come back through the full-text side; paraphrases come back through the vector
side.

In [8]:
from neo4j_graphrag.retrievers import HybridCypherRetriever

hybrid_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name=INDEX_NAME,
    fulltext_index_name=FULLTEXT_INDEX_NAME,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_with_context,
    neo4j_database=NEO4J_DATABASE,
)

results = hybrid_retriever.search(query_text="Nvidia", top_k=3)

for item in results.items:
    print(f"{item.metadata['title']}  ({item.metadata['date']})")
    print(f"  companies: {', '.join(item.metadata['companies'][:4])}")
    print()

NVIDIA MGX Gives System Makers Modular Architecture to Meet Diverse Accelerated Computing Needs of World's Data Centers  (2023-05-29T00:00:00Z)
  companies: QCT, Nvidia Corporation, Super Micro Computer, Inc.

4 Companies Owned by Nvidia  (2023-01-31T00:00:00Z)
  companies: Nvidia Corporation, 3dfx Interactive, Mellanox Technologies

Install drivers for NVIDIA RTX virtual workstations  (2023-04-19T00:00:00Z)
  companies: Nvidia Corporation, Teradici



## 6. The three side by side

Same question, three retrievers. What changes is not which chunks come back - the vector search is
identical - but how much the model gets to see about each one.

In [9]:
QUESTION = "semiconductor manufacturing capacity"

for name, retriever in [
    ("VectorRetriever", vector_retriever),
    ("VectorCypherRetriever", graph_retriever),
    ("HybridCypherRetriever", hybrid_retriever),
]:
    found = retriever.search(query_text=QUESTION, top_k=2)
    print(f"=== {name} ===")
    for item in found.items:
        preview = str(item.content).replace("\n", " ")[:140]
        print(f"  {preview}...")
    print()

=== VectorRetriever ===
  {'text': ', therefore, demand for semiconductor production equipment – is extremely sensitive to market conditions.\nWhen they deteriorate, ...
  {'text': 'Demand for semiconductors and the equipment used in their production has been sky high in recent years, an up-and-up trend industr...

=== VectorCypherRetriever ===
  Semiconductor cycle shows signs of peaking (2022-06-27T00:00:00Z) Companies: Samsung Electronics, Panasonic, Ibiden Competitors of those: no...
  Semiconductor cycle shows signs of peaking (2022-06-27T00:00:00Z) Companies: Samsung Electronics, Panasonic, Ibiden Competitors of those: no...

=== HybridCypherRetriever ===
  Semiconductor cycle shows signs of peaking (2022-06-27T00:00:00Z) Companies: Samsung Electronics, Panasonic, Ibiden Competitors of those: no...
  SMIC subsidiary continues to fulfill Chinese demands for FinFET capabilities (2023-06-07T03:57:55Z) Companies: Semiconductor Manufacturing I...



## 7. Retrievers as agent tools

Three retrievers, three tools, one `deps_type` carrying them. The pattern is the same as the custom
tools in the main notebook: dependencies arrive through `RunContext`, never through the model.

They go into a **`FunctionToolset`** rather than straight onto an agent. A toolset is a named,
reusable bundle, and it can carry its own `instructions` - so the guidance about *which* retriever
suits *which* question travels with the tools instead of being copy-pasted into every agent's
prompt. Pydantic AI appends it to whatever the agent already says.

Two details worth copying:

- **`asyncio.to_thread`.** The retrievers are synchronous and Pydantic AI's tools are async. Calling
  them directly would block the event loop for the length of the query; one wrapper keeps it free.
- **`ModelRetry` on an empty result.** An empty list looks to the model like an answer. A retry
  tells it the search found nothing and to try different wording - which is what a person would do.

In [10]:
import asyncio
from dataclasses import dataclass

from neo4j_graphrag.retrievers.base import Retriever
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.toolsets import FunctionToolset


@dataclass
class RagDeps:
    """The three retrievers, injected per run."""
    vector: Retriever
    graph: Retriever
    hybrid: Retriever


async def _search(retriever: Retriever, query: str, top_k: int) -> list[dict]:
    """Run a synchronous retriever off the event loop."""
    found = await asyncio.to_thread(retriever.search, query_text=query, top_k=top_k)
    if not found.items:
        raise ModelRetry(f"No passages matched '{query}'. Try different wording or broader terms.")
    return [{"text": str(item.content), "metadata": item.metadata} for item in found.items]


retrieval_tools = FunctionToolset(
    id="retrieval",
    instructions=(
        "Choose the retrieval tool that fits the question:\n"
        "- search_news for general or conceptual questions\n"
        "- search_news_with_context when the answer depends on which companies are involved\n"
        "- search_news_hybrid when the question names a specific company, product or figure\n"
        "Answer only from the passages you retrieve."
    ),
)


@retrieval_tools.tool
async def search_news(ctx: RunContext[RagDeps], query: str, top_k: int = 5) -> list[dict]:
    """Semantic search over news article text.

    Args:
        query: What to search for, in natural language.
        top_k: How many passages to return.
    """
    return await _search(ctx.deps.vector, query, top_k)


@retrieval_tools.tool
async def search_news_with_context(ctx: RunContext[RagDeps], query: str, top_k: int = 5) -> list[dict]:
    """Semantic search where each passage arrives expanded through the graph: its article,
    date, sentiment, the companies it mentions and their competitors.

    Args:
        query: What to search for, in natural language.
        top_k: How many passages to return.
    """
    return await _search(ctx.deps.graph, query, top_k)


@retrieval_tools.tool
async def search_news_hybrid(ctx: RunContext[RagDeps], query: str, top_k: int = 5) -> list[dict]:
    """Vector plus full-text search, with the same graph expansion. Use when the question
    contains a specific name, product or number that has to match exactly.

    Args:
        query: What to search for; include the exact term.
        top_k: How many passages to return.
    """
    return await _search(ctx.deps.hybrid, query, top_k)


rag_deps = RagDeps(vector=vector_retriever, graph=graph_retriever, hybrid=hybrid_retriever)
print("Retrieval toolset ready:", retrieval_tools.id)

Retrieval toolset ready: retrieval


### Making the answer checkable

A RAG answer you cannot trace is a guess with the footnotes missing. `output_type` turns the
citation requirement into a schema: the model cannot return an answer without also returning the
articles it rests on, and `grounded` forces it to say plainly whether the passages supported the
claim instead of burying the hedge in prose.

`retriever_used` is in the schema for a different reason - it makes the routing visible, which
section 8 uses.

In [11]:
from typing import Literal

from pydantic import BaseModel, Field


class Source(BaseModel):
    title: str
    date: str | None = None
    article_id: str | None = None


class GroundedAnswer(BaseModel):
    answer: str = Field(description="Two to four sentences, drawn only from retrieved passages.")
    sources: list[Source] = Field(description="The articles the answer rests on.")
    retriever_used: Literal["search_news", "search_news_with_context", "search_news_hybrid"]
    grounded: bool = Field(description="False if the passages did not really support the answer.")


# Both agents share the one toolset - and with it the routing instructions.
rag_agent = Agent(
    MODEL,
    deps_type=RagDeps,
    toolsets=[retrieval_tools],
    instructions="You answer questions about companies using a corpus of news articles.",
)

cited_agent = Agent(
    MODEL,
    deps_type=RagDeps,
    output_type=GroundedAnswer,
    toolsets=[retrieval_tools],
    instructions=(
        "You answer questions about companies using a corpus of news articles. "
        "Cite the articles you used, and set grounded=false if they do not really "
        "support your answer."
    ),
)

print("Two agents, one toolset.")

Two agents, one toolset.


The plain agent first - same tools, prose out. `all_messages()` shows which retriever it reached
for.

In [12]:
result = await rag_agent.run(
    "What is being reported about demand for data centre capacity?",
    deps=rag_deps,
)

print(result.output[:500])
print("\nTools called:", [
    part.tool_name
    for message in result.all_messages()
    for part in message.parts
    if part.part_kind == "tool-call"
])

The reporting suggests demand for data centre capacity is rising, driven by hyperscalers, cloud service providers, enterprises, cloud computing, AI/IoT, and broader digitalisation.

Specific passages mention:
- “growing demand for robust and resilient infrastructure” from hyperscalers, cloud providers, and enterprises
- increased demand for edge data centres and edge computing
- growth in cloud computing bringing more hyperscale investment
- expanding need for cloud access and IoT support

In sh

Tools called: ['search_news_with_context']


Now the same question through the agent with a schema.

In [13]:
result = await cited_agent.run(
    "What is being reported about demand for data centre capacity?",
    deps=rag_deps,
)

answer = result.output
print(answer.answer)
print(f"\nRetriever chosen: {answer.retriever_used}")
print(f"Grounded: {answer.grounded}")
print("\nSources:")
for source in answer.sources:
    print(f"  • {source.title} ({source.date})")

The reporting points to strong and rising demand for data centre capacity, driven by cloud computing, colocation demand, big data, AI/IoT growth and broader digitalisation. One report says the global hyperscale data centre market is driven by rising demand for colocation facilities, surging cloud adoption, and the rapid growth in IoT devices, while another says the market is expanding as governments and businesses increase cloud, big data and edge computing investment. In Germany, the market is also described as continuing to expand, with new land acquired for future data centre development and more facilities planned.

Retriever chosen: search_news_with_context
Grounded: True

Sources:
  • Global Hyperscale Data Center Market to 2032: Surging Adoption of Cloud Technology Fuels the Sector (2023-05-16T23:00:00Z)
  • Central & Eastern Europe Data Center Markets (2019-2024): Leading Players are Equinix, Interxion, IXcellerate, Boosteroid, DEAC, and DataLine (2019-07-29T20:30:00Z)
  • Germ

## 8. Watching the agent choose

Two questions, shaped differently, should land on different tools - a conceptual one on semantic
search, a name-bearing one on hybrid. Because the choice is part of the output schema, you can see
which it picked without reading the message trace.

In [14]:
questions = [
    "How are companies describing their approach to renewable energy?",
    "What has been reported specifically about Nvidia?",
]

for question in questions:
    result = await cited_agent.run(question, deps=rag_deps)
    print(f"Q: {question}")
    print(f"   tool: {result.output.retriever_used}   grounded: {result.output.grounded}")
    print(f"   {result.output.answer[:200]}...")
    print(f"   sources: {len(result.output.sources)}")
    print()

Q: How are companies describing their approach to renewable energy?
   tool: search_news_with_context   grounded: True
   Companies describe their renewable energy approach as a mix of public commitments and practical sourcing strategies. In the retrieved passages, some say they are aiming for 100% renewable power for gl...
   sources: 2

Q: What has been reported specifically about Nvidia?
   tool: search_news_hybrid   grounded: True
   Reportedly, Nvidia has been described as a company that designs, manufactures, and sells graphics processors and related software, and is credited with inventing the GPU. One article says Nvidia expan...
   sources: 3



## 9. When you do not need an agent

Not every retrieval question needs tool choice, a loop, or typed output. `neo4j-graphrag` ships a
`GraphRAG` pipeline that does the straight-line version - retrieve, stuff the context into a
prompt, answer - in three lines.

Use it when the retrieval strategy is fixed. Reach for the agent above when the question has to
decide how to search, when the answer needs a schema, or when retrieval is one step among several.

In [15]:
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.llm import OpenAILLM

pipeline = GraphRAG(
    retriever=graph_retriever,
    llm=OpenAILLM(model_name="gpt-5.4-mini"),
)

response = pipeline.search(
    query_text="What are companies saying about supply chain pressure?",
    retriever_config={"top_k": 5},
    return_context=True,
)

print(response.answer)
print(f"\nRetrieved {len(response.retriever_result.items)} passages.")

Companies are saying supply chain pressure is still a major issue, but it’s starting to ease in some areas.

- **Aptiv’s CFO** said supply chain disruptions **will come down**, implying pressure is improving.
- **Fictiv’s CEO** said the crisis is **not over** and that supply chain disruption is still a critical problem for hardware companies.
- The **Mitto / Retail Insight** data shows shoppers are also feeling the pressure through **out-of-stock items and shipping delays**, with many blaming the pandemic and saying better pay and working conditions could help.

Overall: companies see **ongoing but gradually improving supply chain strain**, with a strong push to **modernize and de-risk** supply chains.

Retrieved 5 passages.


## Cleanup

In [16]:
driver.close()
print("Closed.")

Closed.


## Summary

| Retriever | Searches | Returns | Use when |
| --- | --- | --- | --- |
| `VectorRetriever` | Vector index | Chunk text | The chunk answers the question on its own |
| `VectorCypherRetriever` | Vector index | Whatever your Cypher returns | Context around the chunk matters |
| `HybridRetriever` | Vector + full-text | Chunk text | The question names something exactly |
| `HybridCypherRetriever` | Vector + full-text | Whatever your Cypher returns | Both of the above |

### Key implementation notes

- **The embedder must match the index.** A mismatch returns confident nonsense with no error. Check
  self-similarity once, at the top, and assert on it.
- **`retrieval_query` has `node` and `score` in scope** - that is the whole interface. Everything
  after is ordinary Cypher.
- **Use `result_formatter`.** Without it every item is a driver record repr, awkward to parse and
  poor prompt material.
- **Wrap sync retrievers in `asyncio.to_thread`** before calling them from an async tool.
- **Raise `ModelRetry` on empty results**, so the model rephrases instead of answering from nothing.
- **Bundle tools in a `FunctionToolset` with its own `instructions`.** Routing guidance then lives
  next to the tools and is shared by every agent that mounts them.
- **Put citations in `output_type`.** A schema the model must fill is stronger than an instruction
  it can forget.
- **Pin `neo4j-graphrag>=1.21`.** Older versions emit the `SEARCH` clause without a fallback for
  servers that do not support it.

### Resources

- [neo4j-graphrag documentation](https://neo4j.com/docs/neo4j-graphrag-python/current/)
- [Pydantic AI documentation](https://ai.pydantic.dev/)
- [Main notebook](./pydantic_ai.ipynb) - MCP, custom tools, structured output, approval, memory
- [Aura Agent notebook](./pydantic_ai_aura_agent.ipynb) - a hosted agent over the same graph